In [1]:
from itertools import combinations
import os
import platform
import random

import numpy as np
import pandas as pd
import psutil
from scipy.stats import pearsonr

from ramen.Ramen import Ramen, choose_steps


## Configuration

In [2]:
DATA_PATH = "tutorial_data.csv"
TARGET = "Long Covid"
SEEDS = range(5)

data = pd.read_csv(DATA_PATH)

environment = pd.DataFrame([{
    "OS": platform.system(),
    "processor": platform.processor() or "not reported",
    "logical CPUs": os.cpu_count(),
    "memory GB": round(psutil.virtual_memory().total / 1024**3, 2),
    "samples": data.shape[0],
    "variables": data.shape[1],
}])



display(environment)

,OS,processor,logical CPUs,memory GB,samples,variables
0,Windows,"Intel64 Family 6 Model 186 Stepping 2, Genuine...",16,31.73,400,23


## Repeated-seed experiment

In [3]:
runs = []

for seed in SEEDS:
    print(f"Running seed {seed}")

    random.seed(seed)
    np.random.seed(seed)


    ramen_bowl = Ramen(
        csv_data=DATA_PATH,
        end_string=TARGET,
        min_values=0,
    )

    num_steps = choose_steps(ramen_bowl.df.shape[1])

    ramen_bowl.random_walk(
        num_walks=50000,
        num_steps=num_steps,
        p_value=0.05,
        correction="no_correction",
    )

    ramen_bowl.genetic_algorithm(    
        num_candidates=10,
        reg_factor=0.01,
        hard_stop=100,
    )


    results = ramen_bowl.export_ramen_as_dict()


    runs.append({
        "seed": seed,
        "results": results,
    })


Running seed 0
Starting removing vars with too few values, vectorizing dataframe, and initializing mutual information matrix, this might take a few minutes.
Removed 0 variables because of insufficient data. If deleted too many, please adjust the min_values
done
Optimization terminated successfully.
         Current function value: 4.396451
         Iterations: 12
         Function evaluations: 13
         Gradient evaluations: 13
20
initializing scorer
done initializing scorer
making candidates
Finished making candidates in: 0.04438109998591244
starting Genetic Algorithm
-12.822706515587582
generation: 0 best: -12.495123073490419
generation: 1 best: -12.16215112794621
generation: 2 best: -12.020345101236956
generation: 3 best: -11.903991862143915
generation: 4 best: -11.771476045816929
generation: 5 best: -11.681334746259797
generation: 6 best: -11.591324929284694
generation: 7 best: -11.49295067205178
generation: 8 best: -11.375691007941448
generation: 9 best: -11.31059747248734
gener

## Stability summary

In [4]:
predictors = [column for column in data.columns if column != TARGET]


def arrival_profile(result):
    weights = {
        variable: float(result["END_VAR_ARRIVALS"].get(variable, 0.0))
        for variable in predictors
    }

    total = sum(weights.values())
    if total:
        weights = {
            variable: value / total
            for variable, value in weights.items()
        }

    vector = np.array(
        [weights[variable] for variable in predictors],
        dtype=float,
    )

    ranking = sorted(
        predictors,
        key=lambda variable: (-weights[variable], variable),
    )

    return vector, ranking


def jaccard(left, right):
    union = left | right
    return 1.0 if not union else len(left & right) / len(union)


pairwise_rows = []

for left, right in combinations(runs, 2):
    left_result = left["results"]
    right_result = right["results"]

    left_vector, left_ranking = arrival_profile(left_result)
    right_vector, right_ranking = arrival_profile(right_result)

    left_top10 = set(left_ranking[:10])
    right_top10 = set(right_ranking[:10])

    left_nodes = {
        node
        for edge in left_result["FINAL_NETWORK"]
        for node in edge
    }
    right_nodes = {
        node
        for edge in right_result["FINAL_NETWORK"]
        for node in edge
    }

    pairwise_rows.append({
        "seed_a": left["seed"],
        "seed_b": right["seed"],
        "Pearson r": float(
            pearsonr(left_vector, right_vector).statistic
        ),
        "top-10 overlap": len(left_top10 & right_top10) / 10,
        "active-node Jaccard": jaccard(left_nodes, right_nodes),
    })

pairwise_results = pd.DataFrame(pairwise_rows)
display(pairwise_results.round(3))

,seed_a,seed_b,Pearson r,top-10 overlap,active-node Jaccard
0,0,1,0.901,0.9,0.952
1,0,2,0.889,0.8,0.900
2,0,3,0.934,0.9,1.000
3,0,4,0.900,0.7,0.870
4,1,2,0.942,0.8,0.857
5,1,3,0.953,1.0,0.952
6,1,4,0.917,0.7,0.913
7,2,3,0.939,0.8,0.900
8,2,4,0.931,0.8,0.783
9,3,4,0.925,0.7,0.870


In [5]:
stability_summary = pairwise_results[[
    "Pearson r",
    "top-10 overlap",
    "active-node Jaccard",
]].agg(["mean", "std", "min", "max"]).T
display(stability_summary.round(3))

,mean,std,min,max
Pearson r,0.923,0.021,0.889,0.953
top-10 overlap,0.810,0.099,0.700,1.000
active-node Jaccard,0.900,0.061,0.783,1.000
